# Model Inference and Interpretability
Accompanies the notes on **Model Inference and Interpretability**.

Section 9 needs `shap` and `xgboost` (`pip install shap xgboost`).

## 1. OLS inference by hand, checked with statsmodels

In [ ]:
import numpy as np
import statsmodels.api as sm

x = np.array([1, 2, 3, 4]); y = np.array([2, 3, 5, 6])
ols = sm.OLS(y, sm.add_constant(x)).fit()
print("weights:", ols.params.round(4), " standard errors:", ols.bse.round(4))
print("t:", ols.tvalues.round(3), " p:", ols.pvalues.round(4))
print("95% CI for the slope:", ols.conf_int(alpha=0.05)[1].round(4), " R^2:", round(ols.rsquared, 4))

## 2. Inference on the loan data

In [ ]:
import pandas as pd
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 12)

ld = pd.read_csv("../data/loan_data_train.csv")
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
cols = ["FICO", "Loan.Length", "Amount.Requested", "Inquiries.in.the.Last.6.Months",
        "Monthly.Income", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]
for col in cols:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
ld = ld[cols + ["Interest.Rate"]].dropna()
ld["Amount.Requested"] /= 1000          # in thousands, so the weights are readable
ld["Monthly.Income"] /= 1000
ld["Revolving.CREDIT.Balance"] /= 1000
model = sm.OLS(ld["Interest.Rate"], sm.add_constant(ld[cols])).fit()
table = pd.DataFrame({"weight": model.params, "std error": model.bse, "t": model.tvalues,
                      "p-value": model.pvalues, "CI low": model.conf_int()[0], "CI high": model.conf_int()[1]})
print(table.round(4))
print("R^2:", round(model.rsquared, 4), " n =", int(model.nobs))

## 3. Cohen's f2 for each feature

In [ ]:
r2_full = model.rsquared
for col in cols:
    r2_reduced = sm.OLS(ld["Interest.Rate"], sm.add_constant(ld[[c for c in cols if c != col]])).fit().rsquared
    print(f"{col:32s} f2 = {(r2_full - r2_reduced) / (1 - r2_full):.4f}")

## 4. The p-value trap with a million rows

In [ ]:
rng = np.random.default_rng(0)
n = 1_000_000
x1 = rng.normal(size=n); x2 = rng.normal(size=n)
y_big = 3 * x1 + 0.01 * x2 + rng.normal(size=n)       # x2's effect is tiny
fit = sm.OLS(y_big, sm.add_constant(np.column_stack([x1, x2]))).fit()
r2_without = sm.OLS(y_big, sm.add_constant(x1)).fit().rsquared
print("weight of x2:", fit.params[2].round(4), " p-value:", f"{fit.pvalues[2]:.1e}",
      " f2:", f"{(fit.rsquared - r2_without) / (1 - fit.rsquared):.6f}")

## 5. Logistic regression: odds ratios and marginal effects (bank data)

In [ ]:
bd = pd.read_csv("../data/bd_train.csv")
y_b = (bd["Revenue.Grid"] == 1).astype(int)
feats = ["Online.Purchase.Amount", "Average.Credit.Card.Transaction", "Investment.Tax.Saving.Bond", "Term.Deposit"]
X_b = bd[feats] / 100                    # per 100 units
logit = sm.Logit(y_b, sm.add_constant(X_b)).fit(disp=False)
odds = pd.DataFrame({"weight": logit.params, "odds ratio": np.exp(logit.params),
                     "OR CI low": np.exp(logit.conf_int()[0]), "OR CI high": np.exp(logit.conf_int()[1]),
                     "p-value": logit.pvalues}).iloc[1:]
odds["AME (prob. points)"] = logit.get_margeff(at="overall").margeff
print(odds.round(4))

## 6. MDI vs permutation importance, with two noise columns

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

data = bd.drop(columns=["REF_NO", "post_code", "post_area", "Revenue.Grid"])
X = pd.get_dummies(data, drop_first=True, dtype=float)
X["random_number"] = rng.normal(size=len(X))                  # pure noise, many distinct values
X["random_id"] = rng.integers(0, 1000, len(X)).astype(float)  # pure noise, 1000 levels
x_train, x_test, y_train, y_test = train_test_split(X, y_b, test_size=0.25, random_state=1, stratify=y_b)
rf = RandomForestClassifier(n_estimators=300, max_features=0.3, random_state=0, n_jobs=-1).fit(x_train, y_train)
perm = permutation_importance(rf, x_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=-1)
imp = pd.DataFrame({"MDI (training data)": rf.feature_importances_,
                    "permutation (test AUC drop)": perm.importances_mean}, index=X.columns)
imp["MDI rank"] = imp["MDI (training data)"].rank(ascending=False).astype(int)
imp["perm rank"] = imp["permutation (test AUC drop)"].rank(ascending=False).astype(int)
print(imp.sort_values("MDI (training data)", ascending=False).head(8).round(4))
print(imp.loc[["random_number", "random_id"]].round(4))

## 7. Partial dependence and ICE

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
PartialDependenceDisplay.from_estimator(rf, x_test, ["Online.Purchase.Amount", "Average.Credit.Card.Transaction"],
                                        kind="both", subsample=60, random_state=0, ax=ax,
                                        ice_lines_kw={"alpha": 0.3}, pd_line_kw={"color": "crimson", "lw": 3})
plt.tight_layout(); plt.show()
plt.close("all")

## 8. LIME from scratch for one customer

In [ ]:
from sklearn.linear_model import Ridge

num_cols = ["Online.Purchase.Amount", "Average.Credit.Card.Transaction", "Investment.Tax.Saving.Bond",
            "Life.Insurance", "Term.Deposit", "Personal.Loan"]
x0 = x_test.iloc[[np.argmax(rf.predict_proba(x_test)[:, 1] > 0.5)]]   # the first test customer predicted positive
p0 = rf.predict_proba(x0)[0, 1]
scale = x_train[num_cols].std().values
pert = pd.concat([x0] * 1000, ignore_index=True)
pert[num_cols] = x0[num_cols].values + rng.normal(size=(1000, len(num_cols))) * scale * 0.25   # perturb numeric features
pert[num_cols] = pert[num_cols].clip(lower=0)
p_pert = rf.predict_proba(pert)[:, 1]
z = (pert[num_cols].values - x0[num_cols].values) / scale                   # standardised offsets
weights = np.exp(-(z ** 2).sum(axis=1) / 0.75 ** 2)
surrogate = Ridge(alpha=1.0).fit(z, p_pert, sample_weight=weights)
print(f"model's probability for this customer: {p0:.3f}")
print(pd.Series(surrogate.coef_, index=num_cols).round(4).sort_values())

## 9. Shapley values by hand, then SHAP on XGBoost

In [ ]:
from itertools import combinations
from math import factorial

v = {(): 20, (1,): 30, (2,): 22, (3,): 24, (1, 2): 38, (1, 3): 40, (2, 3): 32, (1, 2, 3): 50}
players = [1, 2, 3]
for j in players:
    others = [k for k in players if k != j]
    phi = 0.0
    for size in range(len(others) + 1):
        for S in combinations(others, size):
            weight = factorial(size) * factorial(len(players) - size - 1) / factorial(len(players))
            phi += weight * (v[tuple(sorted(S + (j,)))] - v[S])
    print(f"phi_{j} = {phi:.4f}")

In [ ]:
import shap
import xgboost as xgb

Xs = X.drop(columns=["random_number", "random_id"])
Xs.columns = Xs.columns.str.replace(r"[\[\]<]", "_", regex=True)
xs_train, xs_test = Xs.loc[x_train.index], Xs.loc[x_test.index]
booster = xgb.XGBClassifier(n_estimators=700, learning_rate=0.05, max_depth=4, subsample=0.8,
                            colsample_bytree=0.8, random_state=0).fit(xs_train, y_train)
explainer = shap.TreeExplainer(booster)
sv = explainer.shap_values(xs_test)                         # one row of values per customer, in log-odds
i = 0
print("base value (log-odds):", round(float(explainer.expected_value), 4))
print("base + sum of SHAP values:", round(float(explainer.expected_value + sv[i].sum()), 4),
      " model's log-odds:", round(float(booster.predict(xs_test.iloc[[i]], output_margin=True)[0]), 4))
print(pd.Series(sv[i], index=Xs.columns).sort_values(key=abs, ascending=False).head(5).round(3))
print("mean |SHAP| (global importance):")
print(pd.Series(np.abs(sv).mean(axis=0), index=Xs.columns).sort_values(ascending=False).head(6).round(3))
plt.figure()
shap.summary_plot(sv, xs_test, max_display=10, show=False)
plt.tight_layout(); plt.show()

## Try it yourself
1. PDP for `Term.Deposit` in section 7.
2. Rerun section 8 with a different seed and bandwidth (0.5 instead of 0.75). How stable is the explanation?